# Probability: Worked Solutions

In [ ]:
import numpy as np


## Problem 1. Conditioning a finite joint law

**Problem.** For the joint table below, compute either conditional row and $P(X=0\mid Y=1)$. Check total expectation and total variance for $Y$. Rows index $X\in\{0,1\}$ and columns index $Y\in\{0,1\}$:

$$
p_{X,Y}=\begin{pmatrix}0.42&0.18\\0.08&0.32\end{pmatrix}.
$$

**Solution.** Dividing each row by its marginal mass gives $P(Y=\cdot\mid X=0)=(0.7,0.3)$ and $P(Y=\cdot\mid X=1)=(0.2,0.8)$. Transposing the table exchanges the conditioning direction, yielding $P(X=0\mid Y=1)=0.18/0.50=0.36$. A row of zero mass cannot define a conditional law.

The conditional means are $0.3$ and $0.8$, with weights $0.6$ and $0.4$. Therefore $E[Y]=0.6(0.3)+0.4(0.8)=0.5$ and

$$
E[\operatorname{Var}(Y\mid X)]=0.19,\qquad
\operatorname{Var}(E[Y\mid X])=0.06.
$$

Their sum $0.25$ equals $\operatorname{Var}(Y)$. The implementation and checks follow.

In [ ]:
def conditional_row(joint, row):
    joint = np.asarray(joint, dtype=float)
    if joint.ndim != 2 or not np.all(np.isfinite(joint)) or np.any(joint < 0):
        raise ValueError("joint must be a finite nonnegative matrix")
    if not np.isclose(joint.sum(), 1.0) or not 0 <= row < joint.shape[0]:
        raise ValueError("normalized joint law and valid row required")
    if joint[row].sum() <= 0:
        raise ValueError("cannot condition on a zero-probability row")
    return joint[row] / joint[row].sum()


In [ ]:
joint = np.array([[0.42, 0.18], [0.08, 0.32]])
np.testing.assert_allclose(conditional_row(joint, 1), [0.2, 0.8])
np.testing.assert_allclose(conditional_row(joint, 0), [0.7, 0.3])
np.testing.assert_allclose(conditional_row(joint.T, 1), [0.36, 0.64])
try:
    conditional_row([[1.0, 0.0], [0.0, 0.0]], 1)
except ValueError:
    pass
else:
    raise AssertionError("zero-probability condition must be rejected")
# Total expectation and variance, computed directly from conditional laws.
px = joint.sum(axis=1)
means = np.array([conditional_row(joint, x)[1] for x in range(2)])
np.testing.assert_allclose(px @ means, 0.5)
within = px @ (means * (1.0 - means))
between = px @ ((means - px @ means) ** 2)
np.testing.assert_allclose([within, between, within + between], [0.19, 0.06, 0.25])
print({"conditional_means": means, "within": within, "between": between})


## Problem 2. Conditioning a Gaussian vector

**Problem.** For a jointly Gaussian column vector, partition the first $k$ coordinates as $X_1$ and observe the remaining coordinates as $X_2=y$. Compute the conditional mean and covariance. Evaluate the scalar example

$$
\mu=(1,-1)^T,\qquad
\Sigma=\begin{pmatrix}4&1.2\\1.2&1\end{pmatrix},\qquad y=0.
$$

**Solution.** Write the covariance in matching blocks $\Sigma_{ij}$. Solving the systems involving $\Sigma_{22}$ gives

$$
E[X_1\mid X_2=y]=\mu_1+\Sigma_{12}\Sigma_{22}^{-1}(y-\mu_2),
$$
$$
\operatorname{Cov}(X_1\mid X_2=y)=\Sigma_{11}-\Sigma_{12}\Sigma_{22}^{-1}\Sigma_{21}.
$$

The scalar answers are $1+1.2(0+1)=2.2$ and $4-1.2^2=2.56$. The code evaluates the inverse products by linear solves. It also checks independent coordinates, a vector-valued first block, and simulated moments of the residual $R=X_1-1-1.2(X_2+1)$.

In [ ]:
def gaussian_conditional(mu, covariance, k, observed):
    mu = np.asarray(mu, dtype=float)
    covariance = np.asarray(covariance, dtype=float)
    observed = np.asarray(observed, dtype=float)
    if mu.ndim != 1 or covariance.shape != (mu.size, mu.size) or not 0 < k < mu.size:
        raise ValueError("incompatible mean, covariance, or partition")
    if observed.shape != (mu.size - k,) or not np.allclose(covariance, covariance.T):
        raise ValueError("observation shape and symmetric covariance required")
    if not all(np.all(np.isfinite(x)) for x in (mu, covariance, observed)):
        raise ValueError("finite inputs required")
    np.linalg.cholesky(covariance)  # This experiment assumes positive definiteness.
    s11, s12 = covariance[:k, :k], covariance[:k, k:]
    s21, s22 = covariance[k:, :k], covariance[k:, k:]
    mean = mu[:k] + s12 @ np.linalg.solve(s22, observed - mu[k:])
    cov = s11 - s12 @ np.linalg.solve(s22, s21)
    return mean, cov


In [ ]:
mu = np.array([1.0, -1.0])
sigma = np.array([[4.0, 1.2], [1.2, 1.0]])
cm, cv = gaussian_conditional(mu, sigma, 1, np.array([0.0]))
np.testing.assert_allclose(cm, [2.2])
np.testing.assert_allclose(cv, [[2.56]])
# Independent Gaussian coordinates: conditioning leaves the first marginal unchanged.
im, iv = gaussian_conditional(mu, np.diag([4.0, 1.0]), 1, [20.0])
np.testing.assert_allclose(im, [1.0])
np.testing.assert_allclose(iv, [[4.0]])
# A vector block checks orientation rather than only the scalar formula.
a = np.array([[2.0, 0.0, 0.0], [0.5, 1.0, 0.0], [1.0, -0.5, 1.0]])
cov3 = a @ a.T
m3, c3 = gaussian_conditional(np.zeros(3), cov3, 2, [0.7])
assert m3.shape == (2,) and c3.shape == (2, 2)
assert np.linalg.eigvalsh(c3).min() > 0
np.testing.assert_allclose(c3, c3.T, atol=1e-14)
# The residual is independent of the observed block in the Gaussian model.
rng = np.random.default_rng(3103)
samples = rng.multivariate_normal(mu, sigma, size=80000)
residual = samples[:, 0] - (1.0 + 1.2 * (samples[:, 1] + 1.0))
assert abs(residual.mean()) < 0.025
assert abs(residual.var() - 2.56) < 0.05
assert abs(np.cov(residual, samples[:, 1])[0, 1]) < 0.025
print({"conditional_mean": cm, "conditional_variance": cv, "residual_variance": residual.var()})


## Problem 3. Monte Carlo means and their standard errors

**Problem.** Estimate $E[U^2]$ for $U\sim\operatorname{Unif}(0,1)$, with an estimated standard error. Compare the variance of $n$-draw estimates at $n=25$ and $n=100$ using $B=12000$ independent repetitions. Also verify Jensen's inequality for the finite law in the code.

**Solution.** Integration gives $E[U^2]=1/3$ and $\operatorname{Var}(U^2)=1/5-1/9=4/45$. For independent observations $V_i=U_i^2$, the estimate and its estimated standard error are

$$
\widehat\mu_n=\frac1n\sum_i V_i,\qquad
\widehat{\operatorname{se}}(\widehat\mu_n)=\frac{s_V}{\sqrt n},\qquad
s_V^2=\frac1{n-1}\sum_i(V_i-\widehat\mu_n)^2.
$$

The exact sampling variance is $4/(45n)$, so quadrupling $n$ divides it by four. The array has one independent dataset per row; averaging along the observation axis produces $B$ estimates whose variance checks this formula.

For the finite law with probabilities $(0.2,0.3,0.5)$ on values $(0.1,1,4)$, concavity of $\log$ implies $\log E[V]-E[\log V]\geq0$. The final calculation evaluates this gap directly.

In [ ]:
def mean_and_standard_error(values):
    values = np.asarray(values, dtype=float)
    if values.ndim != 1 or values.size < 2 or not np.all(np.isfinite(values)):
        raise ValueError("at least two finite observations required")
    return float(values.mean()), float(values.std(ddof=1) / np.sqrt(values.size))


In [ ]:
m, se = mean_and_standard_error([0.0, 1.0, 2.0, 3.0])
np.testing.assert_allclose([m, se], [1.5, np.sqrt(5.0 / 12.0)])
np.testing.assert_allclose(mean_and_standard_error([2.0, 2.0, 2.0]), [2.0, 0.0])
rng = np.random.default_rng(3104)
b = 12000
results = []
for n in (25, 100):
    values = rng.uniform(size=(b, n)) ** 2
    means = values.mean(axis=1)
    empirical_variance = means.var(ddof=1)
    theoretical_variance = 4.0 / (45.0 * n)
    assert abs(empirical_variance / theoretical_variance - 1.0) < 0.06
    example_mean, example_se = mean_and_standard_error(values[0])
    results.append((n, empirical_variance, theoretical_variance, example_se))
    assert abs(means.mean() - 1 / 3) < 0.004
print("n, empirical variance of means, exact variance, SE for one sample")
for row in results:
    print(row)
# Jensen on a specified finite law: log(E X) - E(log X) >= 0.
weights = np.array([0.2, 0.3, 0.5])
positive_values = np.array([0.1, 1.0, 4.0])
gap = np.log(weights @ positive_values) - weights @ np.log(positive_values)
assert gap > 0
print({"jensen_gap": gap})


## Further problems: worked answers

**1. Why can $P(Y=1\mid X=1)$ differ from $P(X=1\mid Y=1)$ even though both use the same joint cell?**

The numerators agree, but the conditioning events have different probabilities. Here the answers are $0.32/0.40=0.8$ and $0.32/0.50=0.64$, respectively. Each conditional probability normalizes a different slice of the table.

**2. Where did Gaussianity enter the argument that the residual is independent of the observed coordinate? Would zero empirical covariance establish independence for arbitrary data?**

The residual and observed coordinate are jointly Gaussian because they are linear functions of a Gaussian vector. Their *population* cross-covariance is zero, which implies independence for jointly Gaussian variables. Without Gaussianity, zero population covariance is insufficient: if $Z$ is uniform on $[-1,1]$, then $Z$ and $Z^2$ are dependent but uncorrelated. A near-zero empirical covariance additionally has sampling error and cannot establish independence by itself.

**3. Increasing $n$ from 25 to 100 divides the variance of the average by four. What changes if only the number of simulation rows $B$ increases?**

The law of each row average remains unchanged. More rows give a more precise empirical description of that law, including a more stable estimate of its variance. Averaging the row averages would define a different estimator using all $Bn$ observations, but that is not what the variance comparison computes.

**4. Compare the spread of individual squared uniforms, the spread of their means, and uncertainty in the simulated estimate of that spread.**

An individual squared uniform has standard deviation $\sqrt{4/45}$. An $n$-observation mean has standard deviation $\sqrt{4/(45n)}$. The standard deviation estimated from $B$ simulated means is itself random; increasing $B$ reduces uncertainty in this estimate. Here all moments exist, so its Monte Carlo fluctuations decrease at the usual $B^{-1/2}$ rate for a fixed $n$. The three quantities concern individual observations, sample means, and a simulation-based summary of sample means, respectively.